# AI-Lake Format — Interactive Demo

Comprehensive walkthrough of the AI-Lake Python SDK:

| Section | What you will learn |
|---|---|
| 1 | **Write** — `open_table()` fluent API, insert, commit |
| 2 | **Load fixture** — pre-generated demo data (500 rows, dim=32) |
| 3 | **Search** — `SearchQuery`, pointer-only results |
| 4 | **Full row data** — `fetch_data=True`, Arrow / Pandas |
| 5 | **Fluent API** — `limit()`, `to_arrow()`, `to_list()`, `to_polars()` |
| 6 | **HNSW tuning** — `pre_normalize`, `normalized_cosine`, `hnsw_m` |
| 7 | **Idempotent writes** — `write_batch_idempotent` |
| 8 | **Iceberg compat** — PyArrow + PyIceberg (no plugin) |
| 9 | **SQL** — DuckDB direct Parquet scan |
| 10 | **RAG context** — `assemble_context()` for LLMs |
| 11 | **Object storage** — MinIO / S3 via PyArrow |

## 0. Setup

In [ ]:
import io
import json
import os
import pathlib

import ailake
import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

TABLE_PATH  = os.environ.get('DEMO_TABLE_PATH', '/data/ailake_demo')
QUERY_PATH  = pathlib.Path(TABLE_PATH).parent / 'demo_query.json'
CUSTOM_PATH = str(pathlib.Path(TABLE_PATH).parent / 'ailake_custom')

print(f'Demo table  : {TABLE_PATH}')
print(f'Custom table: {CUSTOM_PATH}')

## 1. Write — `open_table()` fluent API

`ailake.open_table()` opens or creates an AI-Lake table. The `Table` handle:
- `insert(texts, embeddings)` — buffer a batch (`list[str]` + numpy array or list)
- `commit()` — persist as a new Iceberg snapshot, returns snapshot ID

In Jupyter, `table` renders as an HTML card showing path and vector config.

In [ ]:
# Generate 10 synthetic unit vectors (dim=32 — same as demo fixture)
DIM = 32
rng = np.random.default_rng(seed=42)

topics = [
    "transformer architecture", "retrieval-augmented generation",
    "vector databases", "semantic search", "LLM fine-tuning",
    "approximate nearest neighbor", "columnar storage", "data versioning",
    "stream processing", "knowledge graphs",
]
texts = [f'Custom doc about {t}.' for t in topics]
embs  = rng.standard_normal((len(texts), DIM)).astype(np.float32)
embs /= np.linalg.norm(embs, axis=1, keepdims=True)   # unit vectors

table = ailake.open_table(CUSTOM_PATH, dim=DIM, metric='cosine')
table.insert(texts, embs)
snap_id = table.commit()

print(f'Snapshot ID : {snap_id}')
print(f'Rows written: {len(texts)}')
table   # HTML card in Jupyter

In [ ]:
# Context manager form — equivalent, auto-cleanup
with ailake.open_table(CUSTOM_PATH, dim=DIM, metric='cosine') as t:
    t.insert(['Extra row about caching.'], [embs[0].tolist()])
    snap2 = t.commit()
print(f'Second snapshot: {snap2}')

## 2. Load pre-generated demo fixture

`init_demo.py` wrote 500 rows (dim=32) to `TABLE_PATH` at container start.
It saved the embedding of doc_id=0 to `demo_query.json` so we can verify that
the nearest neighbour of a vector is itself (expected distance ≈ 0).

In [ ]:
with open(QUERY_PATH) as f:
    demo = json.load(f)

DIM_FIXTURE = int(demo['dim'])
query_vec   = np.array(demo['query_vector'], dtype=np.float32)
print(f'Query dim      : {DIM_FIXTURE}')
print(f'Expected top-1 : "{demo["expected_top1_text"][:70]}..."')

## 3. Pointer-only search — `SearchQuery`

`ailake.search()` returns a **`SearchQuery`** — lazy, chainable, no I/O until materialised.

Pointer-only mode (default): `row_id`, `distance`, `file` columns only.
Only the HNSW index is touched; no Parquet column data is loaded.

In [ ]:
results = ailake.search(TABLE_PATH, query_vec, top_k=5)
print(results)          # SearchQuery(5 results, top_k=5)

df_ptr = results.to_pandas()
df_ptr

In [ ]:
# Verify: query vector is its own nearest neighbour (distance ≈ 0)
top = ailake.search(TABLE_PATH, query_vec, top_k=1).to_list()[0]
assert top['distance'] < 0.01, f'Expected near-zero, got {top["distance"]}'
print(f'PASS — distance = {top["distance"]:.2e}  (query found itself)')

## 4. Full row data — `fetch_data=True`

`fetch_data=True` loads all Parquet columns for the matched rows.
Returns a `pyarrow.Table` with all original columns **plus** `_distance` (Float32).

Use when you need the actual text / metadata without a full table scan.

In [ ]:
df_full = ailake.search(TABLE_PATH, query_vec, top_k=5, fetch_data=True).to_pandas()
# Shows all Parquet columns (text, embedding, …) + _distance
df_full[['text', '_distance']]

In [ ]:
arrow_tbl = ailake.search(TABLE_PATH, query_vec, top_k=5, fetch_data=True).to_arrow()
print(f'Schema : {arrow_tbl.schema}')
print(f'Rows   : {arrow_tbl.num_rows}')
arrow_tbl.to_pandas()[['text', '_distance']]

## 5. Fluent API patterns

`SearchQuery` chains: `limit()` → `to_pandas()` / `to_arrow()` / `to_polars()` / `to_list()`.
All materialise lazily on first call; subsequent calls reuse the cached result.

In [ ]:
# .limit() caps k before materialising — still no I/O beyond the index
top3 = ailake.search(TABLE_PATH, query_vec, top_k=10).limit(3).to_pandas()
top3

In [ ]:
# to_list() always returns pointer-only dicts regardless of fetch_data
items = ailake.search(TABLE_PATH, query_vec, top_k=3).to_list()
for i, r in enumerate(items):
    fname = pathlib.Path(r['file']).name
    print(f'  #{i+1}  row_id={r["row_id"]:>4}  distance={r["distance"]:.6f}  file={fname}')

In [ ]:
# to_arrow() pointer-only: pyarrow.Table with row_id, distance, file columns
arrow_ptr = ailake.search(TABLE_PATH, query_vec, top_k=5).to_arrow()
print(f'Columns: {arrow_ptr.schema.names}')
arrow_ptr.to_pandas()

In [ ]:
# to_polars() — polars.DataFrame (pointer-only)
try:
    import polars as pl
    lf = ailake.search(TABLE_PATH, query_vec, top_k=5).to_polars()
    print(f'polars shape: {lf.shape}')
    lf
except ImportError:
    print('polars not installed — pip install polars')

In [ ]:
# Table handle search — search via an already-opened Table object
demo_table = ailake.open_table(TABLE_PATH, dim=DIM_FIXTURE, metric=demo['metric'])
df_handle  = demo_table.search(query_vec, top_k=3, fetch_data=True).to_pandas()
df_handle[['text', '_distance']]

## 6. HNSW tuning — `pre_normalize` + `normalized_cosine`

`pre_normalize=True` normalises vectors to unit-L2 at **write** time and uses
`1 - dot(a, b)` in the HNSW hot loop instead of full cosine (~12-20 % faster
for dim ≥ 1024). Query vectors are normalised automatically at search time.

`metric="normalized_cosine"` disables F16 quantisation (exact F32 stored)
and activates the dot-product hot path.

HNSW graph parameters:

| `hnsw_m` | `hnsw_ef_construction` | Preset |
|---|---|---|
| 8 | 100 | Low latency / high QPS |
| 16 | 150 | General purpose (default) |
| 24 | 200 | High recall (RAG) |
| 32 | 400 | Max recall (medical/legal) |

In [ ]:
NORM_PATH = str(pathlib.Path(TABLE_PATH).parent / 'ailake_normalized')

table_norm = ailake.open_table(
    NORM_PATH,
    dim=DIM,
    metric='normalized_cosine',  # fast 1-dot(a,b); F16 quant disabled
    pre_normalize=True,           # normalise at write time
    hnsw_m=24,
    hnsw_ef_construction=200,
)
table_norm.insert(texts, embs)
table_norm.commit()
print('Normalized table written.')
table_norm

In [ ]:
# Search on normalized table — query is auto-normalised at search time
results_norm = ailake.search(NORM_PATH, embs[0], top_k=3, fetch_data=True).to_pandas()
results_norm[['text', '_distance']]

## 7. Idempotent writes — `write_batch_idempotent`

`TableWriter.write_batch_idempotent(texts, embs, batch_id)` is a no-op if
`batch_id` was already committed. Safe to retry — useful for Airflow re-runs,
at-least-once delivery pipelines, and streaming connectors.

In [ ]:
from ailake import TableWriter

IDEM_PATH   = str(pathlib.Path(TABLE_PATH).parent / 'ailake_idempotent')
extra_texts = ['Idempotent doc A about caching.', 'Idempotent doc B about retries.']
extra_embs  = rng.standard_normal((2, DIM)).astype(np.float32)
extra_embs /= np.linalg.norm(extra_embs, axis=1, keepdims=True)

def _write(batch_id: str) -> int:
    w = TableWriter(IDEM_PATH, dim=DIM, metric='cosine')
    w.write_batch_idempotent(extra_texts, extra_embs.tolist(), batch_id)
    return w.commit()

snap_a = _write('pipeline-run-2026-001')
print(f'First  write : snapshot_id={snap_a}')

snap_b = _write('pipeline-run-2026-001')   # same id — write skipped
print(f'Retry  write : snapshot_id={snap_b}  (batch_id already committed)')

snap_c = _write('pipeline-run-2026-002')   # new id — committed
print(f'New    write : snapshot_id={snap_c}  (new batch_id)')

## 8. Iceberg Compatibility — PyArrow + PyIceberg

The same `.parquet` files are valid Apache Iceberg Spec v2.
Standard readers see all tabular columns; the HNSW footer section is silently ignored.

In [ ]:
# PyArrow — direct Parquet read, zero configuration
parquet_files = sorted(pathlib.Path(TABLE_PATH).rglob('*.parquet'))
parts         = [pq.read_table(str(p)) for p in parquet_files]
arrow_table   = pa.concat_tables(parts)

print(f'Files  : {len(parquet_files)}')
print(f'Rows   : {len(arrow_table)}')
print(f'Schema : {arrow_table.schema}')
arrow_table.to_pandas().head(3)

In [ ]:
# PyIceberg — SqlCatalog backed by SQLite (no Hive/REST required)
from pyiceberg.catalog.sql import SqlCatalog

catalog = SqlCatalog('demo', **{
    'uri'       : 'sqlite:////tmp/demo_iceberg.db',
    'warehouse' : f'file://{TABLE_PATH}',
})

try:
    catalog.create_namespace('default')
except Exception:
    pass

meta_dir = pathlib.Path(TABLE_PATH) / 'default' / 'table' / 'metadata'
hint     = (meta_dir / 'version-hint.text').read_text().strip()
meta_loc = f'file://{meta_dir}/v{hint}.metadata.json'

try:
    catalog.drop_table('default.table')
except Exception:
    pass
catalog.register_table('default.table', meta_loc)

iceberg_tbl = catalog.load_table('default.table')
print(f'Schema: {iceberg_tbl.schema()}')
df_iceberg  = iceberg_tbl.scan().to_pandas()
print(f'Rows  : {len(df_iceberg)}')
df_iceberg.head(3)

## 9. SQL Compatibility — DuckDB

DuckDB reads AI-Lake files as standard Parquet. The `embedding` column appears
as `BLOB` (raw F16 bytes) — the HNSW section in the footer is invisible.

In [ ]:
parquet_glob = str(pathlib.Path(TABLE_PATH) / '**' / '*.parquet')
con = duckdb.connect()

con.execute(
    f"SELECT count(*) AS total_rows, avg(length(text)) AS avg_text_len,"
    f" avg(octet_length(embedding)) AS avg_emb_bytes"
    f" FROM read_parquet('{parquet_glob}', hive_partitioning=false)"
).df()

In [ ]:
# Full-text filter — pushed down to Parquet column scan
con.execute(
    f"SELECT text FROM read_parquet('{parquet_glob}', hive_partitioning=false)"
    f" WHERE text LIKE '%vector search%' LIMIT 5"
).df()

In [ ]:
# Schema inspection — DuckDB shows embedding as BLOB
con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{parquet_glob}', hive_partitioning=false) LIMIT 1"
).df()

## 10. RAG Context Assembly — `assemble_context()`

`ailake.assemble_context()` converts search results into structured XML context
for LLM input. It deduplicates near-identical chunks and respects a token budget.

In [ ]:
full_results = ailake.search(TABLE_PATH, query_vec, top_k=10, fetch_data=True).to_pandas()

chunks = [
    {
        'document_id'  : str(i),
        'chunk_index'  : 0,
        'chunk_text'   : row['text'],
        'document_title': f'Document {i}',
        'distance'     : float(row['_distance']),
    }
    for i, (_, row) in enumerate(full_results.iterrows())
]

context_xml = ailake.assemble_context(
    chunks=chunks,
    max_tokens=2048,
    dedup_threshold=0.1,
)
print(context_xml[:1200])

In [ ]:
# Richer chunk metadata — title, section path, source URI
chunks_rich = [
    {
        'document_id'   : 'doc-001',
        'chunk_index'   : 0,
        'chunk_text'    : 'AI-Lake stores vectors and tabular data in a single Parquet file.',
        'document_title': 'AI-Lake Overview',
        'section_path'  : 'Introduction > Format',
        'source_uri'    : 's3://my-lake/docs/overview.pdf',
        'distance'      : 0.05,
    },
    {
        'document_id'   : 'doc-001',
        'chunk_index'   : 1,
        'chunk_text'    : 'The HNSW index lives in the Parquet footer, invisible to standard readers.',
        'document_title': 'AI-Lake Overview',
        'section_path'  : 'Introduction > Compatibility',
        'source_uri'    : 's3://my-lake/docs/overview.pdf',
        'distance'      : 0.12,
    },
]
print(ailake.assemble_context(chunks=chunks_rich, max_tokens=1024, dedup_threshold=0.05))

## 11. Object Storage — MinIO / S3

Upload the local AI-Lake Parquet files to MinIO (local S3 emulator) and read
them back with PyArrow S3 filesystem — demonstrating object-storage compatibility.

> MinIO console: http://localhost:9001  (user/pass: minioadmin)

In [ ]:
import boto3
from botocore.client import Config

MINIO_ENDPOINT = os.environ.get('MINIO_ENDPOINT', 'http://minio:9000')
ACCESS_KEY     = os.environ.get('MINIO_ACCESS_KEY', 'minioadmin')
SECRET_KEY     = os.environ.get('MINIO_SECRET_KEY', 'minioadmin')
BUCKET         = 'demo-bucket'

s3 = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    config=Config(signature_version='s3v4'),
)

uploaded = []
for local_path in parquet_files:
    key = 'ailake_demo/' + str(local_path.relative_to(TABLE_PATH))
    s3.upload_file(str(local_path), BUCKET, key)
    uploaded.append(key)
    print(f'  -> s3://{BUCKET}/{key}')

print(f'\nTotal: {len(uploaded)} files uploaded')

In [ ]:
import pyarrow.fs as pafs

s3fs = pafs.S3FileSystem(
    endpoint_override=MINIO_ENDPOINT.replace('http://', ''),
    access_key=ACCESS_KEY,
    secret_key=SECRET_KEY,
    scheme='http',
)

s3_tables   = [pq.read_table(f'{BUCKET}/{k}', filesystem=s3fs) for k in uploaded]
s3_combined = pa.concat_tables(s3_tables)

print(f'Rows read from MinIO: {len(s3_combined)}')
s3_combined.to_pandas().head(3)

## Next Steps

| Resource | Location |
|---|---|
| File format binary spec | `docs/specs/FILE_FORMAT.md` |
| Python API reference | `ailake-py/README.md` |
| Iceberg catalog integration | `docs/architecture/CATALOG_BACKENDS.md` |
| DuckDB deep-dive | `02_duckdb.ipynb` |
| Apache Spark | `03_spark.ipynb` |
| Trino (requires `--profile engines`) | `04_trino.ipynb` |
| BigQuery emulator (requires `--profile engines`) | `05_bigquery.ipynb` |